In [0]:
ad_groups_df = spark.read.table("workspace.default.ad_groups_silver")
ads_df = spark.read.table("workspace.default.ads_silver")
ads_reports_daily_df = spark.read.table("workspace.default.ads_reports_daily_silver")
advertisers_df = spark.read.table("workspace.default.advertisers_silver")
campaign_df = spark.read.table("workspace.default.campaign_silver")

In [0]:
from pyspark.sql.functions import *

In [0]:
df_final_adgroups = ad_groups_df.select(
    col("ad_account_id"),
    col("campaign_id"),
    col("ad_group_id"),
    col("optimization_goal"),
    col("billing_event")
)

df_final_ads = ads_df.select(
    col("ad_account_id"),
    col("campaign_id"),
    col("ad_group_id"),
    col("ad_id"),
    col("ad_name"),
    col("ad_text"),
    col("display_name")
)

df_final_ads_reports_daily = ads_reports_daily_df.select(
    col("campaign_id"),
    col("ad_group_id"),
    col("ad_id"),
    col("stat_time_day").alias("date"),
    col("conversion").alias("conversions"),
    col("clicks"),
    col("result").alias("engagement"),
    col("impressions"),
    col("spend"),
    col("purchase"),
    col("shares"),
    col("comments"),
    col("complete_payment"),
    col("clicks_on_music_disc"),
    col("profile_visits"),
    col("total_app_event_add_to_cart"),
    col("registration"),
    col("sales_lead"),
    col("onsite_shopping"),
    col("video_watched_2s").alias("video_views_2s"),
    col("video_watched_6s").alias("video_views_6s"),
    col("video_views_p25").alias("video_views_25p"),
    col("video_views_p50").alias("video_views_50p"),
    col("video_views_p75").alias("video_views_75p"),
    col("video_views_p100").alias("video_views_100p"),
    col("video_play_actions").alias("video_views"),
    col("reach")
)

df_final_advertisers = advertisers_df.select(
    col("ad_account_id"),
    col("ad_account_name")
)

df_final_campaigns = campaign_df.select(
    col("ad_account_id"),
    col("campaign_id"),
    col("campaign_name"),
    col("campaign_status"),
    col("campaign_operation_status"),
    col("objective_type"),
    col("budget")
)

In [0]:
join_df = (
    df_final_ads_reports_daily.alias("jard").join(
    df_final_ads.alias("jads"),
    on = (
        (col("jard.ad_id") == col("jads.ad_id")) &
        (col("jard.ad_group_id") == col("jads.ad_group_id")) &
        (col("jard.campaign_id") == col("jads.campaign_id"))
    ), how ="inner")
    .join(df_final_adgroups.alias("jadg"),
    on = (
        (col("jads.ad_group_id") == col("jadg.ad_group_id")) &
        (col("jads.campaign_id") == col("jadg.campaign_id"))
    ), how="inner")
    .join(df_final_campaigns.alias("jcam"),
    on = (
        (col("jadg.campaign_id") == col("jcam.campaign_id")) &
        (col("jadg.ad_account_id") == col("jcam.ad_account_id"))
    ), how="inner")
    .join(df_final_advertisers.alias("jadv"),
    on = (
        (col("jcam.ad_account_id") == col("jadv.ad_account_id"))
    ), how="inner")
)

In [0]:

df_join_selected = join_df.select(
        col('jadv.ad_account_id'),
        col('jadv.ad_account_name'),
        col('jcam.campaign_id'),
        col('jcam.campaign_name'),
        col('jcam.campaign_status'),
        col('jcam.campaign_operation_status'),
        col('jcam.objective_type'),
        col('jcam.budget'),
        col('jadg.ad_group_id'),
        col('jadg.optimization_goal'),
        col('jadg.billing_event'),
        col('jads.ad_id'),
        col('jads.ad_name'),
        col('jads.ad_text'),
        col('jads.display_name'),
        col('jard.date'),
        col('jard.conversions'),
        col('jard.clicks'),
        col('jard.engagement'),
        col('jard.impressions'),
        col('jard.spend'),
        col('jard.purchase'),
        col('jard.shares'),
        col('jard.comments'),
        col('jard.complete_payment'),
        col('jard.clicks_on_music_disc'),
        col('jard.profile_visits'),
        col('jard.total_app_event_add_to_cart'),
        col('jard.registration'),
        col('jard.sales_lead'),
        col('jard.onsite_shopping'),
        col('jard.video_views_2s'),
        col('jard.video_views_6s'),
        col('jard.video_views_25p'),
        col('jard.video_views_50p'),
        col('jard.video_views_75p'),
        col('jard.video_views_100p'),
        col('jard.video_views'),
        col('jard.reach')
    )


In [0]:
grouped_df = (
    df_join_selected.groupBy(
    col("campaign_id"),
    col("date"),
    col("billing_event")).agg(
    first(col("ad_account_id")).alias("ad_account_id"),
    first(col("ad_account_name")).alias("ad_account_name"),
    first(col("campaign_name")).alias("campaign_name"),
    first(col("campaign_status")).alias("campaign_status"),
    first(col("campaign_operation_status")).alias("campaign_operation_status"),
    first(col("objective_type")).alias("objective_type"),
    first(col("ad_group_id")).alias("ad_group_id"),
    first(col("optimization_goal")).alias("optimization_goal"),
    first(col("ad_id")).alias("ad_id"),
    first(col("ad_name")).alias("ad_name"),
    first(col("ad_text")).alias("ad_text"),
    first(col("display_name")).alias("display_name"),
    sum(col("budget")).alias("budget"),
    sum(col("conversions")).alias("conversions"),
    sum(col("clicks")).alias("clicks"),
    sum(col("engagement")).alias("engagement"),
    sum(col("impressions")).alias("impressions"),
    sum(col("spend")).alias("spend"),
    sum(col("purchase")).alias("purchase"),
    sum(col("shares")).alias("shares"),
    sum(col("comments")).alias("comments"),
    sum(col("complete_payment")).alias("complete_payment"),
    sum(col("clicks_on_music_disc")).alias("clicks_on_music_disc"),
    sum(col("profile_visits")).alias("profile_visits"),
    sum(col("total_app_event_add_to_cart")).alias("total_app_event_add_to_cart"),
    sum(col("registration")).alias("registration"),
    sum(col("sales_lead")).alias("sales_lead"),
    sum(col("onsite_shopping")).alias("onsite_shopping"),
    sum(col("video_views_2s")).alias("video_views_2s"),
    sum(col("video_views_6s")).alias("video_views_6s"),
    sum(col("video_views_25p")).alias("video_views_25p"),
    sum(col("video_views_50p")).alias("video_views_50p"),
    sum(col("video_views_75p")).alias("video_views_75p"),
    sum(col("video_views_100p")).alias("video_views_100p"),
    sum(col("video_views")).alias("video_views"),
    sum(col("reach")).alias("reach"))
)



In [0]:
spark.conf.set("spark.sql.ansi.enabled", "false")
df_final = (
    grouped_df
        .withColumn("frequency", (col("impressions")/col("reach")).cast("Decimal(10,3)"))
        .withColumn("cpm", (col("spend")/col("impressions")*1000).cast("Decimal(10,3)"))
        .withColumn("cpc", (col("spend")/col("clicks")).cast("Decimal(10,3)"))
        .withColumn("cpv", (col("spend")/col("video_views_2s")).cast("Decimal(10,3)"))
        .withColumn("cpe", (col("spend")/col("engagement")).cast("Decimal(10,3)"))
        .withColumn("cpvc", (col("spend")/col("video_views_100p")).cast("Decimal(10,3)"))
        .withColumn("ctr", (col("clicks")/col("impressions")).cast("Decimal(10,3)"))
        .withColumn("vtr", (col("video_views_2s")/col("impressions")).cast("Decimal(10,3)"))
        .withColumn("vtr100", (col("video_views_100p")/col("impressions")).cast("Decimal(10,3)"))
)

In [0]:
df_final.write.format("delta").mode("overwrite").saveAsTable("workspace.default.tiktok_ads_gold")